## Get all the links and save in a excel 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Set up Chrome options (No download functionality needed)
chrome_options = Options()
chrome_options.add_experimental_option('prefs', {
    "plugins.always_open_pdf_externally": True   # Ensure PDFs are handled externally
})

# Initialize WebDriver with ChromeDriverManager and Chrome options
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# List to store all PDF document links
pdf_links = []

# Function to extract links on the current page
def extract_pdf_links():
    try:
        project_links = driver.find_elements(By.CSS_SELECTOR, 'a.viewLink.forPc')
        for link in project_links:
            relative_url = link.get_attribute('href')
            # Ensure the URL is fully qualified
            if not relative_url.startswith("http"):
                report_url = f"https://www.aiib.org{relative_url}"
            else:
                report_url = relative_url
            pdf_links.append(report_url)
            # Print each link as it's extracted in real-time
            print(f"Extracted link: {report_url}")
        print(f"Found {len(project_links)} report links on this page.")
    except Exception as e:
        print(f"Error extracting links: {e}")

# Function to move to the next page
def go_to_next_page():
    try:
        next_button = driver.find_element(By.CSS_SELECTOR, 'a.jp-next')
        next_button.click()  # Go to the next page
        time.sleep(5)  # Wait for the next page to load
        return True
    except Exception as e:
        print(f"No more pages or unable to find 'Next' button: {e}")
        return False

# Step 1: Loop through all pages and extract PDF links
main_url = "https://www.aiib.org/en/projects/list/year/All/member/All/sector/All/financing_type/All/status/Approved" # Main page containing Project Data
driver.get(main_url)
time.sleep(5)  # Wait for the page to load

page = 1
while True:
    print(f"Processing page {page}...")
    extract_pdf_links()  # Extract links on the current page
    
    if not go_to_next_page():  # Try to go to the next page
        break  # No more pages, exit the loop
    
    page += 1

# Quit the driver
driver.quit()

# Step 2: Save extracted links to an Excel file
# Convert the list to a DataFrame
df = pd.DataFrame(pdf_links, columns=['PDF Links'])

# Save DataFrame to Excel
df.to_excel("AIIB.xlsx", index=False)

print("\nLinks have been saved to 'AIIB.xlsx'")


## Download all the documents by specifying the download dir and excel for links 

In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os

# Define download directory
download_dir = ""

# Configure Chrome options for Selenium
chrome_options = Options()
chrome_options.add_experimental_option('prefs', {
    "download.default_directory": download_dir,
    "plugins.always_open_pdf_externally": True,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

# Initialize WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# Function to check if the file is downloaded
def is_file_downloaded(file_name, download_dir):
    file_path = os.path.join(download_dir, file_name)
    return os.path.exists(file_path)

# Function to download documents by clicking on the appropriate link
def download_documents(links):
    for link in links:
        try:
            print(f"Navigating to project page: {link}")
            driver.get(link)  # Navigate to the page with the document link
            
            # Wait for either "Project Document" or "Project Completion Note" link
            project_doc_link = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.XPATH, "//div[@class='link']/a[contains(text(), 'Project Document')]")
                )
            )

            # Extract the filename from the link's href attribute
            file_name = project_doc_link.get_attribute("href").split('/')[-1]
            print(f"Initiating download for: {file_name}")

            # Click the link to trigger download
            project_doc_link.click()

            # Wait for the file to be downloaded
            start_time = time.time()
            while not is_file_downloaded(file_name, download_dir):
                time.sleep(1)
                if time.time() - start_time > 30:  # 30 seconds timeout
                    print(f"Timeout: {file_name} download took too long.")
                    break

            print(f"Downloaded: {file_name}\n")
        
        except Exception as e:
            print(f"Error downloading document from {link}: {e}")

# Read the Excel file and extract links
excel_path = "/Users/adilqasin/Documents/WBG/Scrapping/AIIB/AIIB.xlsx"  # Update with your file path
df = pd.read_excel(excel_path)
extracted_links = df["PDF Links"].dropna().tolist()  # Get links as a list, excluding NaN values

# Start downloading documents
download_documents(extracted_links)

# Quit the driver
driver.quit()


Navigating to project page: https://www.aiib.org/en/projects/details/2024/approved/India-Mumbai-Metro-Line-5.html
Initiating download for: AIIB-Approval-Project-Document-Mumbai-Metro-Line-5-P000365.pdf
Downloaded: AIIB-Approval-Project-Document-Mumbai-Metro-Line-5-P000365.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2020/approved/Bangladesh-Sylhet-to-Tamabil-Road-Upgrade-Project.html
Initiating download for: 20200402-P000153-Sylhet-Tamabil-Road-Upgrade-Published-Document.pdf
Downloaded: 20200402-P000153-Sylhet-Tamabil-Road-Upgrade-Published-Document.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2023/approved/India-Chennai-Peripheral-Ring-Road.html
Initiating download for: AIIB-Chennai-Peripheral-Ring-Road-Sections-2-and-3_PD_Board_Final-20230103.pdf
Downloaded: AIIB-Chennai-Peripheral-Ring-Road-Sections-2-and-3_PD_Board_Final-20230103.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2019/approved/Multicou

Initiating download for: AIIB-P000705-Istanbul-Seismic-Mitigation-and-Emergency-Preparedness-Additional-Financing-Project-TR-ISMEP-AF-APD.pdf
Downloaded: AIIB-P000705-Istanbul-Seismic-Mitigation-and-Emergency-Preparedness-Additional-Financing-Project-TR-ISMEP-AF-APD.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2021/approved/China-Henan-Flood-Emergency-Rehabilitation-and-Recovery-Project.html
Initiating download for: AIIB-P000543_PD-updated-China-Henan-Flood-Emergency-Rehabilitation-and-Recovery-Project_3Mar2022.pdf
Downloaded: AIIB-P000543_PD-updated-China-Henan-Flood-Emergency-Rehabilitation-and-Recovery-Project_3Mar2022.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2021/approved/India-Punjab-Municipal-Services-Improvement-Project.html
Initiating download for: AIIB-Final-Approval-PD-Punjab-Municipal-Services-Improvement-Project_2021April-27.pdf
Downloaded: AIIB-Final-Approval-PD-Punjab-Municipal-Services-Improvement-Project_2021

Initiating download for: AIIB-APD_P000848_Turkiye-Emergency-Road-Rehabilitation-and-Reconstruction-Project.pdf
Downloaded: AIIB-APD_P000848_Turkiye-Emergency-Road-Rehabilitation-and-Reconstruction-Project.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2021/approved/India-Assam-Intra-State-Transmission-System-Enhancement-Project.html
Initiating download for: AIIB-20210129-P000302-India-Assam-Intra-State-Transmission-System-Enhancement-APD-Published.pdf
Downloaded: AIIB-20210129-P000302-India-Assam-Intra-State-Transmission-System-Enhancement-APD-Published.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2019/approved/Sri-Lanka-Reduction-of-Landslide-Vulnerability-by-Mitigation-Measures-Project.html
Initiating download for: P000124-SL-Landslide.pdf
Downloaded: P000124-SL-Landslide.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2024/approved/India-Electric-Bus-Financing-Project.html
Error downloading document fr

Error downloading document from https://www.aiib.org/en/projects/details/2022/approved/Multicountry-Southeast-Asia-Women-s-Economic-Empowerment-Fund.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver                        0x0000000100933f8a chromedriver + 7077770
2   chromedriver                        0x00000001002d50f0 chromedriver + 397552
3   chromedriver                        0x0000000100321383 chromedriver + 709507
4   chromedriver                        0x0000000100321681 chromedriver + 710273
5   chromedriver                        0x0000000100366e14 chromedriver + 994836
6   chromedriver                        0x000000010034593d chromedriver + 858429
7   chromedriver                        0x0000000100364234 chromedriver + 983604
8   chromedriver                        0x00000001003456b3 chromedriver + 857779
9   chromedriver                        0x0000000100314182 chromedriver + 655746
10  chromed

Downloaded: Madhya-Pradesh-Rural-Connectivity-Project.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2024/approved/China-Yantai-Higher-Vocational-School-Project.html
Error downloading document from https://www.aiib.org/en/projects/details/2024/approved/China-Yantai-Higher-Vocational-School-Project.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver                        0x0000000100933f8a chromedriver + 7077770
2   chromedriver                        0x00000001002d50f0 chromedriver + 397552
3   chromedriver                        0x0000000100321383 chromedriver + 709507
4   chromedriver                        0x0000000100321681 chromedriver + 710273
5   chromedriver                        0x0000000100366e14 chromedriver + 994836
6   chromedriver                        0x000000010034593d chromedriver + 858429
7   chromedriver                        0x0000000100364234 chromedriver + 983

...

Initiating download for: Bangladesh-Dhaka-and-Western-Zone-Transmission.pdf
Downloaded: Bangladesh-Dhaka-and-Western-Zone-Transmission.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2019/approved/Bangladesh-Municipal-Water-Supply-and-Sanitation-Project.html
Initiating download for: 20190715-PD-P000068-MWSSP.pdf
Downloaded: 20190715-PD-P000068-MWSSP.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2019/approved/Bangladesh-Power-System-Upgrade-and-Expansion.html
Initiating download for: power-system.pdf
Downloaded: power-system.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2019/approved/Cambodia-Fiber-Optic-Communication-Network-Project.html
Error downloading document from https://www.aiib.org/en/projects/details/2019/approved/Cambodia-Fiber-Optic-Communication-Network-Project.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver        

Initiating download for: 20191212-P000054-India-West-Bengal-published.pdf
Downloaded: 20191212-P000054-India-West-Bengal-published.pdf

Navigating to project page: https://www.aiib.org/en/projects/details/2019/approved/Kazakhstan-Zhanatas-100-MW-Wind-Power-Plant.html
Error downloading document from https://www.aiib.org/en/projects/details/2019/approved/Kazakhstan-Zhanatas-100-MW-Wind-Power-Plant.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver                        0x0000000100933f8a chromedriver + 7077770
2   chromedriver                        0x00000001002d50f0 chromedriver + 397552
3   chromedriver                        0x0000000100321383 chromedriver + 709507
4   chromedriver                        0x0000000100321681 chromedriver + 710273
5   chromedriver                        0x0000000100366e14 chromedriver + 994836
6   chromedriver                        0x000000010034593d chromedriver + 858429
7   

Error downloading document from https://www.aiib.org/en/projects/details/2019/approved/Turkey-Efeler-Geothermal-Power-Plant-Project.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver                        0x0000000100933f8a chromedriver + 7077770
2   chromedriver                        0x00000001002d50f0 chromedriver + 397552
3   chromedriver                        0x0000000100321383 chromedriver + 709507
4   chromedriver                        0x0000000100321681 chromedriver + 710273
5   chromedriver                        0x0000000100366e14 chromedriver + 994836
6   chromedriver                        0x000000010034593d chromedriver + 858429
7   chromedriver                        0x0000000100364234 chromedriver + 983604
8   chromedriver                        0x00000001003456b3 chromedriver + 857779
9   chromedriver                        0x0000000100314182 chromedriver + 655746
10  chromedriver            

Error downloading document from https://www.aiib.org/en/projects/details/2018/approved/India-OSE-InvIT.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver                        0x0000000100933f8a chromedriver + 7077770
2   chromedriver                        0x00000001002d50f0 chromedriver + 397552
3   chromedriver                        0x0000000100321383 chromedriver + 709507
4   chromedriver                        0x0000000100321681 chromedriver + 710273
5   chromedriver                        0x0000000100366e14 chromedriver + 994836
6   chromedriver                        0x000000010034593d chromedriver + 858429
7   chromedriver                        0x0000000100364234 chromedriver + 983604
8   chromedriver                        0x00000001003456b3 chromedriver + 857779
9   chromedriver                        0x0000000100314182 chromedriver + 655746
10  chromedriver                        0x000000010031515

Error downloading document from https://www.aiib.org/en/projects/details/2017/approved/Oman-Oman-Broadband-Infrastructure.html: Message: 
Stacktrace:
0   chromedriver                        0x000000010093bf88 chromedriver + 7110536
1   chromedriver                        0x0000000100933f8a chromedriver + 7077770
2   chromedriver                        0x00000001002d50f0 chromedriver + 397552
3   chromedriver                        0x0000000100321383 chromedriver + 709507
4   chromedriver                        0x0000000100321681 chromedriver + 710273
5   chromedriver                        0x0000000100366e14 chromedriver + 994836
6   chromedriver                        0x000000010034593d chromedriver + 858429
7   chromedriver                        0x0000000100364234 chromedriver + 983604
8   chromedriver                        0x00000001003456b3 chromedriver + 857779
9   chromedriver                        0x0000000100314182 chromedriver + 655746
10  chromedriver                      